### Cell-cell communication (CCC) on the cell2location-mapped lymph node - LIANA+

Continues **cell2location01a_tutorial.ipynb** (human lymph node, 10x Visium **V1_Human_Lymph_Node**, 34 reference cell types from the lymphoid-organ scRNA atlas) and reuses the niche ligand-receptor (LR) library of **squid03_niche_LigRec_programs.ipynb** (**src/programs/niche_lr.py**, **programs.py**).

**Inputs** (written by the tutorial notebook, **results/lymph_nodes_analysis/cell2location_map/**)

- **sp.h5ad** - Visium counts + cell2location posterior: 

  - **obsm['q05_cell_abundance_w_sf']** (cells of each type per spot) and
  -  34 **layers[<cell type>]** = counts of every gene attributed to that cell type (they sum to **X**)
- **region_labels.csv** - Leiden regions of the cell-abundance space (**region_cluster**)
- **data/lymphoid_organ_sc.h5ad** - scRNA reference (for cell-type-resolved LIANA)

#### **Analyses**

1. Load the inferred cell abundances, name the c2l regions

2. Spatial weights (Gaussian kernel, ~2 Visium rings)
3. **LIANA bivariate** - global Moran's R + local cosine score for every LR pair (consensus resource)
4. Which **c2l cell types** host each local LR pattern (Spearman local score vs. cell-type fraction)
5. **LIANA inflow** - source cell type (c2l proportion) & ligand -> receptor
6. **cell2location-resolved sender -> receiver** scores from the per-cell-type count layers
7. **LIANA rank_aggregate** on the scRNA reference, filtered by c2l spatial co-localisation
8. **MISTy** - spatial dependencies between c2l cell types (juxta / para views)
9. **niche_lr + programs** - region niches (center vs. neighbours), biological programs, c2l sender/receiver attribution, LIANA cross-check
10. Consensus table across methods

Helpers: **src/programs/c2l_liana.py** (new), **src/programs/niche_lr.py**, **src/programs/programs.py**.

**Caveats.** A Visium spot (55 um, 100 um centre spacing) holds ~27 cells here (median c2l estimate) and no spot is dominated by one cell type (median dominant fraction 0.17), so spot-level clustering by "dominant cell type" is not used. Every sender/receiver attribution comes from the reference signatures combined with c2l abundances: it says which cell type *can* express L/R *where*, not that it does. The reference has no fibroblastic reticular cells (FRC); stromal chemokines (CCL19/CCL21) are attributed to the available stromal types (VSMC, Endo).



In [ ]:
import os, sys, time, importlib
from importlib.metadata import version
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd

import scipy.sparse as sps
from scipy.stats import rankdata

import anndata as ad
import scanpy as sc
import squidpy as sq
import liana as li

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import seaborn as sns

root0 = Path('/home/flavio/uv/squidpy')
ROOT_SRC = root0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.Basic import create_dir, pdwritecsv, pdreadcsv
from libs.matplotlib_figure_style import apply_figure_style

import programs.niche_lr as nl
import programs.programs as pg
import programs.c2l_liana as cl

print({p: version(p) for p in ['liana', 'scanpy', 'squidpy', 'anndata']})

In [ ]:
root_data    = root0 / 'data'
root_results = root0 / 'results'
root_figure  = root0 / 'figures'

results_folder = root_results / 'lymph_nodes_analysis'
run_name   = results_folder / 'cell2location_map'          # cell2location01a_tutorial output
file_sp    = run_name / 'sp.h5ad'
file_reg   = run_name / 'region_labels.csv'
file_ref   = root_data / 'lymphoid_organ_sc.h5ad'

root_liana = create_dir(results_folder, 'liana')           # tables / h5ad of this notebook
root_fig_liana = create_dir(root_figure, 'c2l_liana')          # figures of this notebook

def _plain(df):        # niche_lr keeps numpy arrays in DataFrame.attrs -> breaks repr / pd.concat
    d = df.copy(); d.attrs = {}; return d

apply_figure_style()

colors = ("#57a4e2", "#ebd246", "#f30707")
cmap_byr = LinearSegmentedColormap.from_list("blue_yellow_red", list(colors))

print(f"file_sp  {file_sp}: {file_sp.exists()}\nfile_reg {file_reg}: {file_reg.exists()}\nfile_ref {file_ref}: {file_ref.exists()}")

### 1. Inferred cells from cell2location

**cl.load_c2l_spots** reads **sp.h5ad** **without** the 34 cell-type layers (~5 GB; they are read lazily, gene by gene, in section 6):
- **var_names** -> gene symbols (LIANA resources use symbols), **var['gene_ids']** keeps Ensembl

- **layers['counts']** = raw counts, **X** = log1p(CP10k)
- **obsm['c2l_abund']** = q05 posterior abundance (*"at least this many cells"*), **obsm['c2l_frac']** = abundance / total

In [ ]:
adata_pos = cl.load_c2l_spots(file_sp, summary='q05', region_labels=file_reg)
type(adata_pos)
adata_pos

In [ ]:
celltypes = list(adata_pos.uns['c2l_factor_names'])
lib_id    = list(adata_pos.uns['spatial'].keys())[0]

df_frac  = adata_pos.obsm['c2l_frac']

print(adata_pos.shape)
print(len(celltypes), 'cell types;', lib_id)
print(f"estimated cells / spot: median {adata_pos.obs.c2l_total_cells.median():.1f}  "
      f"(IQR {adata_pos.obs.c2l_total_cells.quantile(.25):.1f}-{adata_pos.obs.c2l_total_cells.quantile(.75):.1f})")
print(f"dominant cell-type fraction: median {df_frac.max(1).median():.2f}; spots with >= 0.5: {(df_frac.max(1) >= 0.5).sum()}")

In [ ]:
df_abund_perc = adata_pos.obsm['c2l_abund'].copy()
df_abund_perc = df_abund_perc.sum().sort_values(ascending=False).round(0).reset_index()
df_abund_perc.columns = ['cell_type', 'abundance']
df_abund_perc.abundance = df_abund_perc.abundance.astype(int)
total = df_abund_perc.abundance.sum()
df_abund_perc['perc'] = df_abund_perc.abundance / total
print(df_abund_perc.shape)
df_abund_perc

In [ ]:
df_abund = adata_pos.obsm['c2l_abund'].copy()
df_abund.head(3)

#### Region names

The tutorial named regions by the top-3 cell types of the max-scaled mean abundance, which favours rare types (VSMC, DC_pDC appear in many names). Here a region is named by the two cell types with the highest log2 enrichment of their **fraction** (region mean / slide mean), among cell types making up >= 2% of the region.

In [ ]:
print(df_frac.shape)
df_frac.head(6).T.head(4)

In [ ]:
reg_names = cl.region_names_by_enrichment(df_frac, adata_pos.obs['region_cluster'], n=2, min_frac=0.02)
reg_names

In [ ]:
order_reg = sorted(reg_names, key=int)

adata_pos.obs['region_cluster'] = pd.Categorical(adata_pos.obs['region_cluster'].astype(str), categories=order_reg)
adata_pos.obs['region_name'] = adata_pos.obs['region_cluster'].astype(str).map(lambda r: f"{r}: {reg_names[r]}")

df_reg = (pd.DataFrame({'region_cluster': order_reg, 'region_name': [reg_names[r] for r in order_reg],
                        'n_spots': adata_pos.obs['region_cluster'].value_counts().reindex(order_reg).values,
                        'name_tutorial': adata_pos.obs.groupby('region_cluster', observed=True)['region_name_tutorial'].first().reindex(order_reg).values}))

print(df_reg.shape)
print(f"Total spots: {df_reg['n_spots'].sum()}")
df_reg

In [ ]:
filename = root_liana / 'c2l_region_names.tsv'

if not filename.exists():
    pdwritecsv(df_reg, 'c2l_region_names.tsv', root_liana)
    print(f"File saved: {filename}")
else:
    print(f"File already exists: {filename}")

In [ ]:
show = ['B_naive', 'B_mem', 'B_GC_LZ', 'B_GC_DZ', 'FDC', 'T_CD4+_TfH_GC', 'T_CD4+_naive', 'DC_CCR7+',
        'B_plasma', 'Endo', 'VSMC', 'Macrophages_M1']
for c in show:
    adata_pos.obs[f'c2l_{c}'] = df_abund[c].values          # prefixed: avoids clashes with gene names

In [ ]:
%matplotlib inline

fig, axs = plt.subplots(3, 5, figsize=(17, 10.5))
for ax, c in zip(axs.flat, show):
    v = adata_pos.obs[f'c2l_{c}']
    sq.pl.spatial_scatter(adata_pos, color=f'c2l_{c}', library_id=lib_id, img_res_key='hires', img_alpha=0.5,
                          size=1.3, cmap=cmap_byr, vmax=float(np.quantile(v, 0.992)), title=c, ax=ax, colorbar=True)

sq.pl.spatial_scatter(adata_pos, color='region_cluster', library_id=lib_id, img_res_key='hires', img_alpha=0.5,
                      size=1.3, title='c2l regions', ax=axs.flat[len(show)], legend_loc='on data', legend_fontsize=6)

for ax in axs.flat[len(show) + 1:]:
    ax.axis('off')

for ax in axs.flat:
    ax.set_xlabel(''); ax.set_ylabel('')

fig.suptitle('cell2location: q05 cell abundance per spot (colour capped at the 99.2th percentile)', y=0.995)
fig.tight_layout()

filefig = root_fig_liana / 'c2l_abundance_maps.png'
fig.savefig(filefig, dpi=300, bbox_inches='tight')
plt.show()

### 2. Spatial weights

- Coordinates are full-resolution pixels

- the Visium centre-to-centre spacing (100 um) is 137 px. 
- A Gaussian kernel with **bandwidth=150** px (cut at weight 0.1) gives each spot: 
  - itself (1.0, **set_diag=True** - several cells share a spot), 
  - ring 1 (0.65) and 
  - ring 2 (0.18-0.28), 
  - i.e. ~18 neighbours - the same scale as the k<=20 niche of **niche_lr**.

In [ ]:
spacing = nl.spot_spacing(adata_pos.obsm['spatial'])
um_per_px = 100 / spacing
bandwidth = 150

li.ut.spatial_neighbors(adata_pos, bandwidth=bandwidth, cutoff=0.1, kernel='gaussian', set_diag=True)
W = adata_pos.obsp['spatial_connectivities'].tocsr()
W.eliminate_zeros()

i = int(np.argmin(np.linalg.norm(adata_pos.obsm['spatial'] - np.median(adata_pos.obsm['spatial'], 0), axis=1)))   # a central spot
d = np.linalg.norm(adata_pos.obsm['spatial'][W[i].indices] - adata_pos.obsm['spatial'][i], axis=1)

df_w = (pd.DataFrame({'dist_um': (d * um_per_px).round(0), 'weight': W[i].data.round(3)})
        .groupby('dist_um').agg(n=('weight', 'size'), weight=('weight', 'mean')).reset_index())

print(f"spacing {spacing:.1f} px = 100 um -> {um_per_px:.3f} um/px; neighbours per spot (incl. self): {np.diff(W.indptr).mean():.1f}")
df_w

### 3. LIANA bivariate: spatially co-expressed LR pairs

**li.mt.bivariate** on the log-normalised spots with the **consensus** human resource:

- global: **Moran's R** (bivariate spatial autocorrelation of L and R, permutation p)

- local: **cosine** similarity of L and R within each spot's weighted neighbourhood, with **cats** (high-high, high-low, ...) and local permutation p-values

**nz_prop=0.05**: L and R (all complex subunits) expressed in >= 5% of spots.
<br>

In [ ]:
filename_libiv = root_liana / 'visium_lymph_node_liana_bivariate.h5ad'

force=False

if filename_libiv.exists() and not force:
    adata_lr = ad.read_h5ad(filename_libiv)
    print(f"read {filename_libiv}")
else:
    t0 = time.time()
    adata_lr = li.mt.bivariate(adata_pos, resource_name='consensus', local_name='cosine', global_name='morans',
                             n_perms=100, mask_negatives=False, add_categories=True, nz_prop=0.05,
                             use_raw=False, verbose=True)
    adata_lr.write_h5ad(filename_libiv)
    print(f"bivariate done in {time.time() - t0:.0f} s -> {filename_libiv}")

fname = 'liana_bivariate_global.tsv'
filename = root_liana / fname

if not filename.exists():
    df_inter = adata_lr.var.sort_values('morans', ascending=False).copy()

    df_inter['morans_fdr'] = nl._bh(df_inter.morans_pvals.values)          # BH over all tested pairs

    pdwritecsv(df_inter.reset_index(), fname, root_liana)
    print(f"written {filename}")
else:
    df_inter = pdreadcsv(fname, root_liana)
    df_inter.set_index('interaction', inplace=True)
    print(f"read {filename}")

df_inter_sig = df_inter[(df_inter.morans_fdr < 0.05) & (df_inter.morans > 0)]

print(f"LR pairs tested: {len(df_inter)};  Moran's R > 0 at FDR < 0.05: {len(df_inter_sig)};  R > 0.05: {(df_inter_sig.morans > 0.05).sum()}")

cols = ['ligand', 'receptor', 'ligand_props', 'receptor_props', 'morans', 'morans_fdr']
df_inter[cols].head(20)


In [ ]:
top_show = df_inter.index[:6].tolist()

fig = plt.figure(figsize=(16, 7.5))
gs = fig.add_gridspec(2, 4, width_ratios=[1.25, 1, 1, 1])
ax = fig.add_subplot(gs[:, 0])
t25 = df_inter.head(25)[::-1]
ax.barh(t25.index, t25.morans, color='#4c72b0')
ax.set_xlabel("global Moran's R (bivariate)")
ax.set_title('top 25 LR pairs')
ax.tick_params(axis='y', labelsize=6)

for k, lr_id in enumerate(top_show):
    ax = fig.add_subplot(gs[k // 3, 1 + k % 3])
    v = adata_lr[:, lr_id].X
    v = np.asarray(v.toarray() if sps.issparse(v) else v).ravel()
    # local cosine saturates near 1 where L and R are both broadly expressed -> percentile limits for contrast
    sq.pl.spatial_scatter(adata_lr, color=lr_id, library_id=lib_id, img_res_key='hires', img_alpha=0.4,
                          size=1.3, cmap=cmap_byr, vmin=float(np.quantile(v, 0.02)), vmax=float(np.quantile(v, 0.995)),
                          title=f"{lr_id}  (local cosine, 2-99.5th pct)", ax=ax)
    ax.set_xlabel(''); ax.set_ylabel('')
fig.tight_layout()

plt.show()

filefig = root_fig_liana / 'liana_bivariate_top.png'
fig.savefig(filefig, dpi=300, bbox_inches='tight')

### 4. Which c2l cell types host the spatial LR patterns?

Spearman correlation across spots between the **local** LR score and each cell type's **fraction** (c2l). High df_rho = the interaction is strongest where that cell type is enriched - a host / niche, not proof that this cell type is the sender or receiver.

In [ ]:
def _rz(M):
    R = np.apply_along_axis(rankdata, 0, M)
    return (R - R.mean(0)) / (R.std(0) + 1e-12)

In [ ]:
lr_top = df_inter_sig.index[:40]
Xl = adata_lr[:, lr_top].X
Xl = Xl.toarray() if sps.issparse(Xl) else np.asarray(Xl)
fontsize=12

df_rho = pd.DataFrame(_rz(Xl).T @ _rz(df_frac.values) / len(Xl), index=lr_top, columns=celltypes)

g = sns.clustermap(df_rho, cmap='RdBu_r', center=0, vmin=-0.8, vmax=0.8, figsize=(15, 14),
                   dendrogram_ratio=(0.06, 0.06), 
                   # cbar_kws={'label': 'Spearman rho'},
                   xticklabels=True, yticklabels=True)

ax = g.ax_heatmap

# Place legend at the extreme right outside the axes
# Place colorbar at the extreme right, outside the axes (beyond the row labels)
fig = g.fig
fig.canvas.draw()                                   # needed so label extents are known
renderer = fig.canvas.get_renderer()

bboxes = [t.get_window_extent(renderer) for t in ax.get_yticklabels()]
bboxes.append(ax.yaxis.label.get_window_extent(renderer))
x_right_px = max(b.x1 for b in bboxes)
x_right = fig.transFigure.inverted().transform((x_right_px, 0))[0]   # pixels -> figure coords

hm = ax.get_position()
pad, width = 0.09, 0.015
g.ax_cbar.set_position([x_right + pad,              # left
                        hm.y0 + 0.30 * hm.height,   # bottom (vertically centred)
                        width,                      # width
                        0.40 * hm.height])          # height

# Tick font size (heatmap + colorbar) — must come before measuring label extents
g.ax_cbar.tick_params(labelsize=fontsize)
g.ax_cbar.yaxis.set_ticks_position('right')
g.ax_cbar.yaxis.set_label_position('right')

ax.set_xlabel('c2l cell type (fraction)')
ax.set_ylabel('LR pair (local cosine)')
ax.xaxis.label.set_size(fontsize + 4)
ax.yaxis.label.set_size(fontsize + 4)

ax.tick_params(axis='x', labelsize=fontsize)    # cell-type names
ax.tick_params(axis='y', labelsize=fontsize)   # LR-pair names

g.fig.suptitle("Spearman rho heatmap", fontsize=fontsize + 6, y=1.02)
plt.show()

filefig = root_fig_liana / 'liana_local_vs_c2l_fraction.png'
g.savefig(filefig, dpi=300, bbox_inches='tight')

In [ ]:
df_rho.head(3)

In [ ]:
fname = 'liana_local_vs_c2l_fraction_spearman.tsv'
filename = root_liana / fname

if not filename.exists():
    pdwritecsv(df_rho.reset_index(names='interaction'), fname, root_liana)
    print(f"File saved to {filename}")

In [ ]:
pd.set_option('display.max_colwidth', 300)

df_rho.apply(lambda r: ', '.join(f"{c} ({r[c]:.2f})" for c in r.nlargest(8).index), axis=1).head(15).to_frame('top host cell types')

In [ ]:
root_liana

### Claude - Opus 5.5

From the table above, can you infer biological functions and save a table at root_liana?

I'll infer a function for each of these 15 pairs from three things:
- **Program membership:** which **programs.py** programs contain the ligand or receptor.
- **Program activity at the pair's hotspots:** Spearman correlation between the pair's local score and each program's activity across spots.
- **Context from the earlier sections:** host cell types, the best cell2location sender→receiver, and the inflow source.

I'll add a short, literature-based interpretation per pair, with confidence flags where the pairing is questionable.

Program correlations align with the expected host cell types
- CXCL13/FCER2/LTB with MHC-II/humoral-B, 
- CCR7 with MHC-I/cytotoxic T-NK, and
- ACKR1/TIMP1 with chemotaxis/angiogenesis/ECM programs. 

I'll now put together the summary table.

I've saved a table of inferred biological functions for the 15 pairs (15 rows × 16 columns) to **/home/flavio/uv/squidpy/results/lymph_nodes_analysis/liana/ccc_lr_biological_functions.tsv**.

Each function combines a data-driven part with a short interpretation from established lymph-node immunology:
- **Programs at the hotspots:** the 3 programs from **programs.py** (out of 51) whose activity rises most where the pair's local score is high, measured as Spearman ρ across spots.
- **Programs containing the ligand or receptor:** from **nl.annotate_lr_programs**.
- **Context from the notebook:** the top host cell types (your table), the top 3 cell2location sender→receiver pairs, and the inflow-enriched sources.
- **Interpretation:** a lymph-node compartment, a function statement, a confidence level and a caveat.

| Compartment | Pairs | Inferred function | Programs active at hotspots |
|---|---|---|---|
| T-cell zone | CCL21→CCR7, CCL19→CCR7 | Homing of naive T cells, Tregs and DCs; DC–T cell encounters (stromal CCL21; DC_CCR7+ CCL19) | MHC-I antigen presentation, cytotoxic T/NK, chemotaxis (ρ 0.27–0.35) |
| B follicle / germinal centre | FCER2→CR2, CXCL13→CXCR5 | B cell–FDC adhesion and antigen retention; CXCL13 from FDC attracting B cells and TfH into follicles and the GC light zone | MHC-II antigen presentation (0.40–0.43), humoral B cell |
| Vessels / medulla | CCL14→ACKR1, CCL2→ACKR1, CCL2→CCR2 | ACKR1 on venules presenting and clearing chemokines; recruitment of CCR2+ monocytes | Chemotaxis, angiogenesis, phagocytosis |
| Plasma-cell niche | THBS1→SDC1 | Plasma-cell adhesion and survival in the medullary cords | ER stress/UPR (0.27), as expected for secretory cells |
| Stroma / myeloid | TIMP1→CD63, THY1→ITGAX/ITGB2 | ECM remodelling and survival signalling; myeloid cells adhering to stroma | ECM remodelling, EMT, TNF/NF-κB |
| Follicle edge | CCL20→CCR6 | Positioning of CCR6+ memory and pre-GC B cells | MHC-II, ECM |

I marked 4 pairs as **low confidence**:
- **LCK→CD8A/CD8B** is an interaction inside the same cell, so its signal mainly reflects how many CD8 T cells are present.
- **LTB→CD40** is not the canonical pairing; lymphotoxin normally signals through LTBR.
- **FN1→CD79A** has weak direct evidence. Read it as fibronectin matrix next to B cells rather than a signal through the B-cell receptor.
- **VIM→CD44** involves genes expressed almost everywhere, so its local score is high in most spots.

The cell2location sender assignments follow the reference expression profiles. Because the reference has no fibroblastic reticular cells, CCL21/CCL19 are assigned to VSMC, endothelium and DC_CCR7+ instead.


In [ ]:
fname = 'ccc_lr_biological_functions.tsv'
df_lr_bio = pdreadcsv(fname, root_liana)

df_lr_bio.head(3).T

### 5. LIANA inflow: source cell type & ligand -> receptor

**li.mt.inflow(obsm_key='c2l_frac')** builds, for each spot, the trivariate score **W . (ligand x proportion of source) x receptor** -> features **source^ligand^receptor**. Restricted to the 150 LR pairs with the highest global Moran's R.

Because the spot ligand is multiplied by the source **proportion**, raw inflow is dominated by abundant cell types (B_mem, B_naive, T_CD4+_naive). The table therefore also gives the source **enrichment** = log2(share of the LR total carried by the source / mean fraction of the source): > 0 when the source is over-represented around the places where L->R is high.

In [ ]:
file_inf = root_liana / 'liana_inflow_source_enrichment.tsv'
lr150 = list(zip(df_inter.ligand[:150], df_inter.receptor[:150]))

force=False

if file_inf.exists() and not force:
    df_inf = pd.read_csv(file_inf, sep='\t')
else:
    inf = li.mt.inflow(adata_pos, obsm_key='c2l_frac', interactions=lr150, use_raw=False)
    df_inf = cl.inflow_source_enrichment(inf, df_frac)
    pdwritecsv(df_inf, file_inf.name, root_liana)
    del inf

In [ ]:
import programs.c2l_liana as cl
importlib.reload(cl)

In [ ]:
fontsize = 12
lr_inf = df_inter.index[:25]
E_inf = (df_inf[df_inf.interaction.isin(lr_inf)]
         .pivot(index='interaction', columns='source', values='log2_enrichment').reindex(lr_inf))

g = sns.clustermap(E_inf.fillna(0), cmap='RdBu_r', center=0, vmin=-1.5, vmax=1.5, figsize=(15, 10),
                   row_cluster=False, dendrogram_ratio=(0.06, 0.08), xticklabels=True, yticklabels=True)
cl.style_clustermap(g, xlabel='source cell type (c2l proportion)', ylabel="LR pair (ordered by Moran's R)",
                    title='LIANA inflow: source enrichment', cbar_label='log2 enrichment', fontsize=fontsize)
plt.show()

g.savefig(root_fig_liana / 'liana_inflow_source_enrichment.png', dpi=300, bbox_inches='tight')

In [ ]:
df_inf[df_inf.interaction.isin(lr_inf[:10])].groupby('interaction', sort=False).head(2)[
    ['interaction', 'source', 'global_mean', 'share', 'log2_enrichment']].round(3)

### 6. cell2location-resolved sender -> receiver

cell2location splits every spot's counts across the 34 cell types (**layers[<cell type>]**, sum = **X**). With those layers the ligand can be taken from the **sender** cell type and the receptor from the **receiver** cell type:

**score[A,B] = mean_spots( (W_norm . L_A) x R_B )**, with **L_A**, **R_B** = log1p(CP10k) of the counts attributed to A / B (complex = min over subunits) and **W_norm** the row-normalised kernel of section 2.

Null: receptor spots permuted 200 times -> z and p (Gaussian tail of z when < 5 permutations reach the score), BH-FDR. The spot-permutation null breaks all spatial structure, so **any** co-localised pair is significant; rank pairs by **score** / **score_share** (share of the LR total) and z, not by FDR alone.

The 100 LR pairs with the highest Moran's R (FDR < 0.05) are scored (100 x 34 x 34 = 115,600 sender-receiver-LR tests).

In [ ]:
file_ct = root_liana / 'c2l_celltype_resolved_lr.tsv'
lr100 = df_inter_sig.head(100)[['ligand', 'receptor']]

force=False

if file_ct.exists() and not force:
    df_ct = pd.read_csv(file_ct, sep='\t')
else:
    t0 = time.time()
    genes_lr = sorted({s for e in list(lr100.ligand) + list(lr100.receptor) for s in e.split('_')})
    E, genes_E, _ = cl.celltype_expr(file_sp, adata_pos, genes_lr, celltypes)        # spots x cell types x genes
    df_ct = cl.celltype_resolved_lr(E, genes_E, celltypes, lr100, W, n_perms=200, seed=0)
    pdwritecsv(df_ct, file_ct.name, root_liana)
    del E
    print(f"{df_ct.interaction.nunique()} LR pairs scored in {time.time() - t0:.0f} s")

best_ct = (df_ct[df_ct.rank_in_lr == 1].set_index('interaction')
           .reindex([i for i in df_inter_sig.index[:100] if i in set(df_ct.interaction)]))

cols = ['sender', 'receiver', 'score', 'score_share', 'z', 'fdr']
best_ct[cols].head(25).round(3)

In [ ]:
lr_panels = [x for x in ['CCL21^CCR7', 'CCL19^CCR7', 'CXCL13^CXCR5', 'FCER2^CR2', 'CCL2^ACKR1', 'LTB^CD40']
             if x in set(df_ct.interaction)]

fig, axs = plt.subplots(3, 2, figsize=(17, 17))
for ax, lr_id in zip(axs.flat, lr_panels):
    Mx = cl.lr_matrix(df_ct, lr_id, 'score')
    s_top = Mx.sum(1).nlargest(10).index
    r_top = Mx.sum(0).nlargest(10).index
    sns.heatmap(Mx.loc[s_top, r_top], ax=ax, cmap='rocket_r', cbar_kws={'label': 'score', 'shrink': .7},
                xticklabels=True, yticklabels=True)
    ax.set_title(lr_id); ax.set_ylabel('sender (ligand)'); ax.set_xlabel('receiver (receptor)')
for ax in axs.flat[len(lr_panels):]:
    ax.axis('off')
fig.suptitle('cell2location-resolved CCC: top 10 senders x top 10 receivers per LR pair', y=1.0)
fig.tight_layout()
fig.savefig(root_fig_liana / 'c2l_celltype_resolved_lr_panels.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# strongest sender -> receiver links over all LR pairs (top 40 by score, z > 5)
top40 = df_ct[df_ct.z > 5].nlargest(40, 'score').iloc[::-1]
lab = top40.sender + ' -> ' + top40.receiver + '   ' + top40.interaction.str.replace('^', ' -> ', regex=False)

fig, ax = plt.subplots(figsize=(7.5, 10))
sc_ = ax.scatter(top40.score, np.arange(len(top40)), c=top40.z, s=40 + 300 * top40.score_share, cmap='viridis')
ax.set_yticks(np.arange(len(top40))); ax.set_yticklabels(lab, fontsize=6)
ax.set_xlabel('score  (mean of niche ligand x receptor)')
plt.colorbar(sc_, ax=ax, shrink=.5, label='z vs spot permutation')
ax.set_title('top sender -> receiver links (size = share of LR total)')
fig.tight_layout()
fig.savefig(root_fig_liana / 'c2l_celltype_resolved_top_links.png', dpi=300, bbox_inches='tight')
plt.show()

### 7. LIANA rank_aggregate on the scRNA reference + c2l co-localisation

Classical, cell-type-resolved LIANA (CellPhoneDB, Connectome, log2FC, NATMI, SingleCellSignalR, CellChat-like; **rank_aggregate**) on the reference used to build the c2l signatures (**Subset**, 34 types; up to 400 cells per type, log1p(CP10k)). It is non-spatial, so each source -> target pair is annotated with the **c2l co-localisation** of the two cell types: Pearson r between the abundance of A at a spot and the kernel-weighted abundance of B around it (symmetrised).

"Spatially supported" = **specificity_rank <= 0.05**, **magnitude_rank <= 0.05** and co-localisation > 0.2 (with 100 permutations the specificity rank of most specific pairs bottoms out near 0.03, so 0.01 would keep only a handful).

In [ ]:
df_abund

In [ ]:
file_rank = root_liana / 'liana_rank_aggregate_reference.tsv'

force=False

if file_rank.exists() and not force:
    liana_ref = pd.read_csv(file_rank, sep='\t')
else:
    t0 = time.time()
    refb = ad.read_h5ad(file_ref, backed='r')
    obs_ref = refb.obs[['Subset', 'Method', 'Sample', 'Tissue']]
    rng = np.random.default_rng(0)
    keep = np.sort(np.concatenate([rng.choice(ix, size=min(400, len(ix)), replace=False)
                                   for ix in obs_ref.groupby('Subset', observed=True).indices.values()]))
    ref = refb[keep].to_memory(); refb.file.close()
    ref = ad.AnnData(X=ref.X.astype(np.float32), obs=ref.obs[['Subset', 'Method', 'Sample', 'Tissue']].copy(),
                     var=pd.DataFrame(index=ref.var_names))          # var_names = symbols in the original file
    sc.pp.normalize_total(ref, target_sum=1e4); sc.pp.log1p(ref)
    print(ref.shape, ref.obs.Subset.value_counts().min(), 'cells in the smallest type')
    li.mt.rank_aggregate(ref, groupby='Subset', resource_name='consensus', expr_prop=0.1, use_raw=False,
                         n_perms=100, n_jobs=8, verbose=False)
    liana_ref = ref.uns['liana_res']
    pdwritecsv(liana_ref, file_rank.name, root_liana)
    del ref
    print(f"rank_aggregate in {time.time() - t0:.0f} s")

coloc = cl.spatial_colocalisation(df_abund, W)
pdwritecsv(coloc.reset_index(names='cell_type'), 'c2l_celltype_colocalisation.tsv', root_liana)

liana_ref['coloc'] = [coloc.at[s, t] for s, t in zip(liana_ref.source, liana_ref.target)]
sel = liana_ref[(liana_ref.specificity_rank <= 0.05) & (liana_ref.magnitude_rank <= 0.05)]
sup = sel[sel.coloc > 0.2].sort_values('specificity_rank')
pdwritecsv(sup, 'liana_rank_aggregate_spatially_supported.tsv', root_liana)
print(f"reference interactions: {len(liana_ref)};  specific & strong: {len(sel)};  + co-localised (r > 0.2): {len(sup)}")
sup.sort_values('magnitude_rank')[['source', 'target', 'ligand_complex', 'receptor_complex', 'magnitude_rank', 'specificity_rank', 'coloc']].head(20).round(4)

In [ ]:
fontsize = 12
g = sns.clustermap(coloc, cmap='RdBu_r', center=0, vmin=-0.8, vmax=0.8, figsize=(15, 14),
                   dendrogram_ratio=(0.06, 0.06), xticklabels=True, yticklabels=True)
cl.style_clustermap(g, xlabel='cell type', ylabel='cell type',
                    title='c2l cell-type co-localisation (spot vs. neighbourhood abundance)',
                    cbar_label='co-localisation r', fontsize=fontsize)
plt.show()

g.savefig(root_fig_liana / 'c2l_celltype_colocalisation.png', dpi=300, bbox_inches='tight')

Germinal-centre compartment (FDC, GC B, GC TfH, tingible-body-like macrophages - cell types that co-localise in section 7), spatially supported interactions only:

In [ ]:
gc_types = ['FDC', 'T_CD4+_TfH_GC', 'B_GC_LZ', 'B_GC_DZ', 'B_GC_prePB', 'B_Cycling', 'Macrophages_M1']
sup_gc = sup[sup.source.isin(gc_types) & sup.target.isin(gc_types)]
print(f"{len(sup_gc)} spatially supported interactions within the GC compartment")
p = li.pl.dotplot(liana_res=sup_gc, colour='magnitude_rank', size='specificity_rank',
                  inverse_colour=True, inverse_size=True,
                  source_labels=[c for c in gc_types if c in set(sup_gc.source)],
                  target_labels=[c for c in gc_types if c in set(sup_gc.target)],
                  top_n=30, orderby='magnitude_rank', orderby_ascending=True,
                  figure_size=(18, 8), return_fig=True)
p.save(root_fig_liana / 'liana_rank_aggregate_GC_dotplot.png', dpi=300, verbose=False)
p

### 8. MISTy: spatial dependencies between c2l cell types

**li.mt.genericMistyData** on the c2l **fractions**: each cell type is predicted from the other cell types in its **juxta** view (6 nearest spots = ring 1) and **para** view (Gaussian, bandwidth 275 px ~ 2 rings). **bypass_intra=True**: fractions of one spot sum to 1, so the intra view would predict a target trivially.

**gain_R2** = variance explained by the spatial context; the importances show which neighbouring cell types predict each target.

In [ ]:
comp = ad.AnnData(X=df_frac.values.astype(np.float32), 
                  obs=adata_pos.obs[['region_cluster']].copy(),
                  var=pd.DataFrame(index=celltypes), 
                  obsm={'spatial': adata_pos.obsm['spatial']})

misty = li.mt.genericMistyData(intra=comp, set_diag=False, cutoff=0.05, bandwidth=275, n_neighs=6, verbose=False)
misty(model=li.mt.sp.RandomForestModel, bypass_intra=True, n_jobs=8, verbose=False)

tm = misty.uns['target_metrics'].sort_values('gain_R2', ascending=False)
pdwritecsv(tm, 'misty_c2l_target_metrics.tsv', root_liana)
pdwritecsv(misty.uns['interactions'], 'misty_c2l_interactions.tsv', root_liana)
tm[['target', 'multi_R2', 'gain_R2', 'juxta', 'para']].head(12).round(3)

In [ ]:
fig1 = li.pl.contributions(misty, figure_size=(6, 8), return_fig=True)
fig1.save(root_fig_liana / 'misty_c2l_contributions.png', dpi=300, verbose=False)      # plotnine figure
fig1

In [ ]:
imp = (misty.uns['interactions'].query("view == 'juxta'")
       .pivot(index='predictor', columns='target', values='importances').fillna(0))
g = sns.clustermap(imp, cmap='mako_r', figsize=(12, 11), dendrogram_ratio=(0.08, 0.08), vmax=float(np.quantile(imp.values, 0.99)),
                   cbar_kws={'label': 'importance (juxta)'}, xticklabels=True, yticklabels=True)
g.ax_heatmap.set_xlabel('target cell type'); g.ax_heatmap.set_ylabel('predictor (neighbouring spots)')
g.savefig(root_fig_liana / 'misty_c2l_juxta_importances.png', dpi=300, bbox_inches='tight')
plt.show()

### 9. Niche LR + biological programs (**niche_lr**, **programs**) on c2l regions

As in **squid03_niche_LigRec_programs.ipynb**, now with the human consensus resource and the human **PROGRAMS** / **CELL_STATES**:
- one centre per c2l region (**nl.centers_by_group**: most interior spot), here the **germinal-centre** region (highest B_GC_LZ + FDC fraction)
- centre pooled with ring 1 (**center_pool=6**), partners = remaining 14 spots; background = 2,000 niches centred **outside** the region
- both directions (center->neighbors, neighbors->center), z / FDR vs background, gradients along 3-6 collinear spots, program activity and inferred functions

New here (c2l): the niche's **cell-type composition** vs background, and the **sender / receiver cell types** of each niche LR = share of the ligand (receptor) counts attributed to each cell type in the c2l layers.

In [ ]:
centers = nl.centers_by_group(adata_pos, groupby='region_cluster')
gc_score = (df_frac['B_GC_LZ'] + df_frac['FDC']).groupby(adata_pos.obs['region_cluster'].astype(str)).mean()
region_gc = gc_score.idxmax()
center = centers[region_gc]
bg_mask = (adata_pos.obs['region_cluster'].astype(str) != region_gc).values
lr_h = nl.load_lr('consensus')

print(f"GC region: {region_gc} ({reg_names[region_gc]}), center spot {adata_pos.obs_names[center]} (idx {center}); LR resource {lr_h.shape}")

res = nl.analyze_niche(adata_pos, center, k=20, center_pool=6, lr=lr_h, programs=pg.PROGRAMS,
                       bg_mask=bg_mask, layer=None, fdr=0.1, n_top_gradients=15, n_line_cells=6,
                       um_per_unit=um_per_px, n_background=2000)
pdwritecsv(res.lr, f'niche_lr_region{region_gc}.tsv', root_liana)
pdwritecsv(res.functions, f'niche_functions_region{region_gc}.tsv', root_liana)

df_comp = cl.niche_composition(adata_pos, res, bg_mask=bg_mask)
df_comp.head(8).round(3)

In [ ]:
cols = ['ligand', 'receptor', 'direction', 'score', 'z', 'fdr', 'ligand_sender', 'receptor_receiver',
        'ligand_programs', 'receptor_programs']
_plain(res.lr)[cols].head(12).round(4)

In [ ]:
# which c2l cell types carry the ligand (sender side) and the receptor (receiver side)?
df_att = cl.niche_celltype_attribution(file_sp, adata_pos, res, n=15)
pdwritecsv(df_att, f'niche_lr_region{region_gc}_c2l_attribution.tsv', root_liana)
df_att.round(3)

In [ ]:
res.functions[['category', 'program', 'z', 'percentile', 'lr_support', 'n_lr', 'top_lr', 'score']].head(12).round(3)

In [ ]:
df_states = nl.niche_program_activity(adata_pos, center, res.neighbors, pg.CELL_STATES, n_background=2000, bg_mask=bg_mask)
df_states.head(10).round(3)

In [ ]:
# LIANA cross-check: local cosine at the centre / neighbours and global rank (Moran's R) of the niche LRs
df_chk = nl.liana_local_check(adata_lr, res, n=20)
df_chk.round(3)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1, 1.1]})
nl.plot_niche(adata_pos, res, ax=axs[0])
sq.pl.spatial_scatter(adata_pos, color='region_cluster', library_id=lib_id, img_res_key='hires', img_alpha=0.5, size=1.3,
                      groups=[region_gc], na_color='#e0e0e0', title=f'region {region_gc}: {reg_names[region_gc]}', ax=axs[1])
xy = adata_pos.obsm['spatial'][center] * adata_pos.uns['spatial'][lib_id]['scalefactors']['tissue_hires_scalef']
axs[1].scatter(*xy, marker='*', s=200, c='crimson', zorder=5)
axs[1].set_xlabel(''); axs[1].set_ylabel('')
fig.tight_layout()
fig.savefig(root_fig_liana / f'niche_map_region{region_gc}.png', dpi=300, bbox_inches='tight')
plt.show()

if len(res.gradients):
    keys = res.gradients.sort_values(['consistent_with_direction', 'z'], ascending=False).key.head(6).tolist()
    fig = nl.plot_gradients(res, keys)
    fig.savefig(root_fig_liana / f'niche_gradients_region{region_gc}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
res.gradients[['key', 'n_cells', 'L_rho', 'R_rho', 'pattern', 'consistent_with_direction', 'z', 'fdr']].round(3)

#### All c2l regions: program activity per region niche

**nl.scan_niches**: one niche per region, each compared with niches centred outside its own region (1,000 background niches).

In [ ]:
file_mat = root_liana / 'niche_functions_by_c2l_region.tsv'

t0 = time.time()
df_mat, niche_results = nl.scan_niches(adata_pos, {r: centers[r] for r in order_reg if r in centers}, value='z',
                                       exclude_own_group='region_cluster', center_pool=6, lr=lr_h,
                                       programs=pg.PROGRAMS, um_per_unit=um_per_px, n_background=1000,
                                       n_top_gradients=5)
df_mat.columns = [f"{c}: {reg_names[c]}" for c in df_mat.columns]
pdwritecsv(df_mat.reset_index(), file_mat.name, root_liana)
print(f"{len(niche_results)} region niches in {time.time() - t0:.0f} s")

top = df_mat.loc[df_mat.abs().max(1).sort_values(ascending=False).index[:30]]
g = sns.clustermap(top, cmap='RdBu_r', center=0, vmin=-5, vmax=5, figsize=(13, 10),
                   dendrogram_ratio=(0.08, 0.08), cbar_kws={'label': 'niche program activity z'},
                   xticklabels=True, yticklabels=True)
g.savefig(root_fig_liana / 'niche_programs_by_c2l_region.png', dpi=300, bbox_inches='tight')
plt.show()

### 10. Consensus across methods

Per LR pair (top 100 by Moran's R): global spatial co-expression (bivariate), best c2l-resolved sender -> receiver, most enriched inflow source, best spatially supported reference pair (rank_aggregate + co-localisation) and the number of region niches where the pair is significant (niche_lr FDR <= 0.1).

In [ ]:
ids = [i for i in df_inter_sig.index[:100]]
inf_best = df_inf.sort_values('log2_enrichment', ascending=False).groupby('interaction').head(1).set_index('interaction')

sup_ = sup.assign(interaction=sup.ligand_complex + '^' + sup.receptor_complex).sort_values('magnitude_rank')
sup_best = sup_.groupby('interaction').head(1).set_index('interaction')

niche_sig = pd.concat([_plain(r.lr).assign(region=k)[lambda d: d.fdr <= 0.1] for k, r in niche_results.items()])
niche_sig['interaction'] = niche_sig.ligand + '^' + niche_sig.receptor
n_niche = niche_sig.groupby('interaction').region.nunique()
niche_regions = niche_sig.groupby('interaction').region.apply(lambda s: ','.join(sorted(set(s), key=int)))

cons = pd.DataFrame(index=pd.Index(ids, name='interaction'))
cons['morans'] = df_inter.morans.reindex(ids)
cons['morans_fdr'] = df_inter.morans_fdr.reindex(ids)
cons['c2l_sender_receiver'] = (best_ct.sender + ' -> ' + best_ct.receiver).reindex(ids)
cons['c2l_score_share'] = best_ct.score_share.reindex(ids)
cons['c2l_z'] = best_ct.z.reindex(ids)
cons['inflow_top_source'] = inf_best.source.reindex(ids)
cons['inflow_log2_enrich'] = inf_best.log2_enrichment.reindex(ids)
cons['ref_source_target'] = (sup_best.source + ' -> ' + sup_best.target).reindex(ids)
cons['ref_magnitude_rank'] = sup_best.magnitude_rank.reindex(ids)
cons['ref_coloc'] = sup_best.coloc.reindex(ids)
cons['n_region_niches'] = n_niche.reindex(ids).fillna(0).astype(int)
cons['niche_regions'] = niche_regions.reindex(ids).fillna('')
cons['n_lines_of_evidence'] = (1 + cons.ref_source_target.notna().astype(int)
                               + (cons.n_region_niches > 0).astype(int) + (cons.inflow_log2_enrich > 0.3).astype(int))

pdwritecsv(cons.reset_index(), 'ccc_consensus_c2l_liana.tsv', root_liana)
cons.sort_values(['n_lines_of_evidence', 'morans'], ascending=False).head(25).round(3)

### Notes for interpretation

- **Spot level.** Bivariate / inflow / niche scores are computed on ~27-cell spots; a high score means L and R co-vary in space at ~100-200 um, which also happens when their producers merely share a compartment.
- **Attribution = reference x abundance.** c2l layers split each spot's counts in proportion to (cell abundance x reference signature). Sender/receiver calls therefore follow the reference signatures (e.g. a gene detected in the FDC reference cluster is assigned to FDC wherever FDC are present) and cannot reveal cell-type expression that is absent from the reference.
- **Missing stroma.** The reference has no FRC / fibroblast types; CCL19/CCL21/CXCL12 counts are assigned to VSMC / Endo / DC_CCR7+.
- **Nulls.** The c2l-resolved spot-permutation null and the bivariate permutation p-values ignore spatial autocorrelation, so most co-localised pairs pass; use them to rank, and treat FDR as liberal.
- **Single section.** One lymph node section from one donor; no replicate-level inference is possible.